In [50]:
# ===== 0. SETUP & FUNGSI BANTU =====
import os, re, io, json, time, gzip, hashlib, datetime as dt
import requests, pandas as pd, numpy as np
from urllib import robotparser
from urllib.parse import urlparse

BASE = "UTS-DAM"
for d in ["data/raw", "data/processed", "data/output", "docs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

KONTAK = "isi-email-kelompok@contoh.ac.id"   # <-- GANTI dengan email kelompok
UA_NAME = "UAS-DesaWisata-Mabar"
HEADERS = {"User-Agent": f"{UA_NAME}/1.0 (tugas kuliah; {KONTAK})"}
TODAY = dt.date.today().isoformat()

PROV = []       # catatan provenance
HTTP_LOG = []   # catatan status HTTP / retry (bukti Soal 3)

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def log_source(nama, url, path, lisensi, catatan=""):
    PROV.append({"nama": nama, "url": url, "tanggal_akses": TODAY, "lisensi": lisensi,
                 "file": path, "ukuran_byte": os.path.getsize(path), "md5": md5_file(path), "catatan": catatan})

def fetch(url, params=None, method="get", data=None, max_retry=5, timeout=90, label=""):
    """HTTP request dengan penanganan status & exponential backoff."""
    for i in range(1, max_retry + 1):
        try:
            r = requests.request(method, url, params=params, data=data, headers=HEADERS, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError) as e:
            HTTP_LOG.append({"label": label, "percobaan": i, "status": f"ERR {type(e).__name__}"})
            time.sleep(2 ** i); continue
        HTTP_LOG.append({"label": label, "percobaan": i, "status": r.status_code})
        if r.status_code == 200:
            return r
        if r.status_code in (429, 500, 502, 503, 504):      # sementara -> retry
            wait = int(r.headers["Retry-After"]) if r.headers.get("Retry-After", "").isdigit() else 2 ** i
            print(f"[{label}] HTTP {r.status_code} -> retry {i}/{max_retry} dalam {wait}s"); time.sleep(wait); continue
        r.raise_for_status()                                  # 4xx lain -> gagal langsung
    raise RuntimeError(f"Gagal setelah {max_retry} percobaan: {url}")

def norm(s):
    """Normalisasi nama wilayah untuk kunci merge."""
    s = str(s).lower()
    s = re.sub(r"\[[^\]]*\]", "", s)
    s = re.sub(r"\b(kecamatan|kelurahan|desa|kampung|kab\.?|kabupaten)\b", "", s)
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
print("Siap. Tanggal akses:", TODAY)

Siap. Tanggal akses: 2026-11-26


In [51]:
# ============================================================
# SOAL 1 — DATA RESMI DESA WISATA BERKELANJUTAN KEMENPAREKRAF
# Tidak menggunakan data dummy, data sintetis, atau mirror tidak
# terverifikasi. Jika sumber resmi gagal, proses dihentikan.
# ============================================================

import requests
import pandas as pd
from pathlib import Path
import json
from datetime import datetime, timezone

URL1_PAGE = (
    "https://katalogdata.kemenpar.go.id/dataset/"
    "jumlah-desa-wisata-berkelanjutan-tersertifikasi-tahun-2025"
)
RESOURCE_ID = "0119776f-6fbd-4961-ac5d-5039d4336b1a"
API_BASE = "https://katalogdata.kemenpar.go.id/api/3/action"

def get_json_or_raise(url, params=None, timeout=60):
    """Ambil JSON dari portal resmi; gagal -> hentikan, bukan buat data palsu."""
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    if payload.get("success") is not True:
        raise RuntimeError(
            "Portal Kemenparekraf mengembalikan status success=False: "
            + str(payload.get("error", payload))
        )
    return payload

# 1. Verifikasi metadata resource pada portal resmi
resource_meta = get_json_or_raise(
    f"{API_BASE}/resource_show",
    params={"id": RESOURCE_ID}
)["result"]

print("Sumber resmi :", URL1_PAGE)
print("Nama resource:", resource_meta.get("name"))
print("Format       :", resource_meta.get("format"))
print("Resource ID  :", RESOURCE_ID)

# 2. Unduh seluruh record dengan pagination (bukan hanya 1.000/2.000 baris)
records = []
limit = 1000
offset = 0

while True:
    result = get_json_or_raise(
        f"{API_BASE}/datastore_search",
        params={
            "resource_id": RESOURCE_ID,
            "limit": limit,
            "offset": offset
        }
    )["result"]

    batch = result.get("records", [])
    records.extend(batch)
    total = result.get("total")

    print(f"Diunduh: {len(records)}" + (f" / {total}" if total is not None else " baris"))

    if not batch:
        break
    offset += len(batch)
    if total is not None and len(records) >= total:
        break

# 3. Validasi: tidak boleh mengganti kegagalan unduh dengan data buatan
if not records:
    raise RuntimeError(
        "Tidak ada record yang diterima dari resource resmi Kemenparekraf. "
        "Periksa koneksi, status portal, dan Resource ID. Tidak ada data dummy dibuat."
    )

df1 = pd.DataFrame(records)
df1.columns = [str(c).strip().lower() for c in df1.columns]

# Validasi kolom minimum yang dipakai oleh proses integrasi berikutnya.
# Jika skema portal berubah, hentikan proses dan tampilkan kolom aktual;
# jangan mengarang/mengisi kolom yang tidak tersedia.
required_columns = ["kabupaten_kota"]
missing_columns = [c for c in required_columns if c not in df1.columns]
if missing_columns:
    raise ValueError(
        "Skema resource resmi tidak memiliki kolom wajib "
        f"{missing_columns}. Kolom yang tersedia: {df1.columns.tolist()}. "
        "Periksa dokumentasi resource dan sesuaikan pemetaan kolom berdasarkan "
        "metadata resmi; notebook sengaja tidak membuat nilai pengganti."
    )

# Simpan respons asli dan metadata sumber agar provenance bisa diaudit
RAW1 = Path(BASE) / "data" / "raw" / "kemenpar_desa_wisata_berkelanjutan_2025.csv"
RAW1.parent.mkdir(parents=True, exist_ok=True)
df1.to_csv(RAW1, index=False, encoding="utf-8-sig")

RAW1_JSON = RAW1.with_suffix(".json")
with open(RAW1_JSON, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

metadata_kemenpar = {
    "nama_dataset": resource_meta.get("name"),
    "portal": URL1_PAGE,
    "api_endpoint": f"{API_BASE}/datastore_search",
    "resource_id": RESOURCE_ID,
    "jumlah_record_diunduh": len(df1),
    "waktu_pengambilan_utc": datetime.now(timezone.utc).isoformat(),
    "status": "data aktual dari API resmi Kemenparekraf",
    "catatan": (
        "Tidak ada record yang dibuat, direkayasa, atau diisi dengan data dummy. "
        "Jumlah baris mengikuti record yang tersedia pada resource saat pengambilan."
    )
}
with open(RAW1.parent / "metadata_kemenpar_2025.json", "w", encoding="utf-8") as f:
    json.dump(metadata_kemenpar, f, ensure_ascii=False, indent=2)

# Catat provenance dengan fungsi notebook jika tersedia
if "log_source" in globals():
    log_source(
        "Kemenparekraf - Desa Wisata Berkelanjutan Tersertifikasi 2025",
        URL1_PAGE,
        str(RAW1),
        "Lisensi mengikuti ketentuan yang tercantum pada portal sumber; "
        "periksa halaman dataset sebelum redistribusi."
    )

print("\\nJumlah baris x kolom:", df1.shape)
print("Jumlah record aktual:", len(df1))
print("Kolom:", df1.columns.tolist())
print("\\nTipe data:")
print(df1.dtypes)
print("\\nCakupan wilayah jika kolom tersedia:")
for col in ["provinsi", "kabupaten_kota", "kabupaten", "kecamatan"]:
    if col in df1.columns:
        print(f"- {col}: {df1[col].nunique()} nilai unik")
print("\\nFile tersimpan:")
print("-", RAW1)
print("-", RAW1_JSON)
print("-", RAW1.parent / "metadata_kemenpar_2025.json")

# Penting: jangan memaksakan jumlah minimum dengan membuat baris palsu.
if len(df1) < 100:
    print(
        "\\nPERHATIAN: data resmi yang tersedia kurang dari 100 baris. "
        "Dataset tetap dipertahankan apa adanya. Jangan menambahkan data dummy. "
        "Jika tugas mensyaratkan >=100 baris, cari resource resmi lain yang "
        "relevan dan dokumentasikan sumbernya sebelum digabungkan."
    )

display(df1.head())


Sumber resmi : https://katalogdata.kemenpar.go.id/dataset/jumlah-desa-wisata-berkelanjutan-tersertifikasi-tahun-2025
Nama resource: Jumlah Desa Wisata/ Destinasi dengan Penerapan Standar Pariwisata Berkelanjutan Tahun 2025
Format       : CSV
Resource ID  : 0119776f-6fbd-4961-ac5d-5039d4336b1a
Diunduh: 10 / 10
\nJumlah baris x kolom: (10, 12)
Jumlah record aktual: 10
Kolom: ['_id', 'tahun_awal_sertifikasi', 'desa_wisata', 'kode_desa_kelurahan', 'desa_kelurahan', 'kode_kecamatan', 'kecamatan', 'kode_kabupaten_kota', 'kabupaten_kota', 'kode_provinsi', 'provinsi', 'masa_berlaku']
\nTipe data:
_id                        int64
tahun_awal_sertifikasi     int64
desa_wisata               object
kode_desa_kelurahan        int64
desa_kelurahan            object
kode_kecamatan             int64
kecamatan                 object
kode_kabupaten_kota        int64
kabupaten_kota            object
kode_provinsi              int64
provinsi                  object
masa_berlaku              object
dtype: o

,_id,tahun_awal_sertifikasi,desa_wisata,kode_desa_kelurahan,desa_kelurahan,kode_kecamatan,kecamatan,kode_kabupaten_kota,kabupaten_kota,kode_provinsi,provinsi,masa_berlaku
0,1,2025,Desa Wisata Wanurejo,3308020010,Desa Wanurejo,3308020,Kecamatan Borobudur,3371,Kabupaten Magelang,33,Jawa Tengah,2025-2028
1,2,2025,Desa Wisata Krebet,3402160002,Kelurahan Sendangsari,3402160,Kecamatan Pajangan,3402,Kabupaten Bantul,34,DI Yogyakarta,2025-2028
2,3,2025,Desa Wisata Les,5108090007,Desa Les,5108090,Kecamatan Tejakula,5108,Kabupaten Buleleng,51,Bali,2025-2028
3,4,2025,Desa Wisata Senaru,5208050005,Desa Senaru,5208050,Kecamatan Bayan,5208,Kabupaten Lombok Utara,52,Nusa Tenggara Barat,2025-2028
4,5,2025,Desa Wisata Pentingsari,3404170005,Desa Umbulharjo,3404170,Kecamatan Cangkringan,3404,Kabupaten Sleman,34,DI Yogyakarta,2025-2028


**filter tiga kabupaten dan estimasi penyimpanan**

In [52]:
# Filter Manggarai Raya (pencocokan nama kabupaten secara fleksibel)
KAB = {"manggarai barat": "Manggarai Barat", "manggarai": "Manggarai", "manggarai timur": "Manggarai Timur"}
df1["key_kab"] = df1["kabupaten_kota"].astype(str).apply(norm)
df1_raya = df1[df1["key_kab"].isin(KAB.keys())].copy()
df1_raya["kabupaten"] = df1_raya["key_kab"].map(KAB)
print("Baris Manggarai Raya:", len(df1_raya), "| >= 100 baris:", len(df1) >= 100)
if len(df1_raya) > 0:
    print(df1_raya["kabupaten"].value_counts().to_string())
display(df1_raya.head(10))

# Estimasi penyimpanan tahunan (asumsi dapat diubah)
size_kb = os.path.getsize(RAW1) / 1024
gz_kb = len(gzip.compress(open(RAW1, "rb").read())) / 1024
SNAPSHOT_PER_TAHUN = 1      # dirilis/diperbarui 1x per tahun (ganti 12 bila bulanan)
BUFFER = 1.2                # +20% metadata/log/versi
est = pd.DataFrame({
    "skenario": ["CSV mentah", "CSV terkompres gzip"],
    "per_snapshot_KB": [size_kb, gz_kb],
    "estimasi_per_tahun_KB": [size_kb * SNAPSHOT_PER_TAHUN * BUFFER, gz_kb * SNAPSHOT_PER_TAHUN * BUFFER]})
print(f"Rumus: ukuran_snapshot x {SNAPSHOT_PER_TAHUN} snapshot/tahun x buffer {BUFFER}")
est.round(2)

Baris Manggarai Raya: 0 | >= 100 baris: False


,_id,tahun_awal_sertifikasi,desa_wisata,kode_desa_kelurahan,desa_kelurahan,kode_kecamatan,kecamatan,kode_kabupaten_kota,kabupaten_kota,kode_provinsi,provinsi,masa_berlaku,key_kab,kabupaten


Rumus: ukuran_snapshot x 1 snapshot/tahun x buffer 1.2


,skenario,per_snapshot_KB,estimasi_per_tahun_KB
0,CSV mentah,1.44,1.73
1,CSV terkompres gzip,0.59,0.71


**pemindaian PII dan quasi-identifier**

In [53]:
PII_PATTERNS = {
    "email":   r"[\w\.-]+@[\w\.-]+\.[a-z]{2,}",
    "telepon": r"(?:\+62|62|0)8\d{7,12}",
    "nik_16d": r"\b\d{16}\b",
}
def scan_pii(df, nama):
    hasil = []
    for col in df.select_dtypes(include="object").columns:
        s = df[col].dropna().astype(str)
        for jenis, pat in PII_PATTERNS.items():
            n = int(s.str.contains(pat, regex=True, flags=re.I).sum())
            if n: hasil.append({"dataset": nama, "kolom": col, "jenis_pii": jenis, "jumlah_temuan": n})
    return pd.DataFrame(hasil, columns=["dataset", "kolom", "jenis_pii", "jumlah_temuan"])

pii_s1 = scan_pii(df1, "Kemenpar-CSV")
print("Temuan PII pada dataset Soal 1:", len(pii_s1)); display(pii_s1)

QUASI = [c for c in df1.columns if c in ("desa_wisata", "desa_kelurahan", "kecamatan", "kabupaten_kota", "provinsi", "tahun_awal_sertifikasi")]
print("Kandidat quasi-identifier:", QUASI)
print("Catatan: semua berupa data wilayah/entitas (bukan individu); risiko re-identifikasi rendah. Tidak ada nama orang/kontak pribadi.")

Temuan PII pada dataset Soal 1: 0


,dataset,kolom,jenis_pii,jumlah_temuan


Kandidat quasi-identifier: ['tahun_awal_sertifikasi', 'desa_wisata', 'desa_kelurahan', 'kecamatan', 'kabupaten_kota', 'provinsi']
Catatan: semua berupa data wilayah/entitas (bukan individu); risiko re-identifikasi rendah. Tidak ada nama orang/kontak pribadi.


**ambil data via API Overpass (OpenStreetMap)**

In [54]:
OVERPASS = "https://overpass-api.de/api/interpreter"
BBOX = {"Manggarai Barat": (-9.05, 119.65, -8.20, 120.50),
        "Manggarai": (-8.85, 120.15, -8.35, 120.65),
        "Manggarai Timur": (-8.90, 120.40, -8.25, 121.00)}

def q_area(nama):
    return ('[out:json][timeout:120];\n'
            'area["boundary"="administrative"]["name"="Kabupaten %s"]->.a;\n'
            '(node["place"~"^(village|hamlet|suburb|town|island)$"](area.a);node["tourism"](area.a););\n'
            'out tags center;') % nama
def q_bbox(b):
    s = "%s,%s,%s,%s" % b
    return ('[out:json][timeout:120];\n(node["place"~"^(village|hamlet|suburb|town|island)$"](%s);'
            'node["tourism"](%s););\nout tags center;') % (s, s)

elems = []
for nama, bb in BBOX.items():
    print(f"Mengambil data untuk {nama}...")
    try:
        js = fetch(OVERPASS, method="post", data={"data": q_area(nama)}, label=f"S3-{nama}").json()
        metode = "area"
    except Exception as e:
        print(f"{nama}: Gagal mengambil data area ({str(e)}), mencoba fallback bbox...")
        js = fetch(OVERPASS, method="post", data={"data": q_bbox(bb)}, label=f"S3-{nama}-bbox").json()
        metode = "bbox"

    if len(js.get("elements", [])) < 10:
        print(f"{nama}: hasil area terlalu sedikit ({len(js.get('elements', []))}), mencoba fallback bounding box...")
        js = fetch(OVERPASS, method="post", data={"data": q_bbox(bb)}, label=f"S3-{nama}-bbox").json()
        metode = "bbox"

    raw = f"{BASE}/data/raw/osm_overpass_{nama.lower().replace(' ', '_')}.json"
    json.dump(js, open(raw, "w", encoding="utf-8"), ensure_ascii=False)
    log_source(f"OpenStreetMap via Overpass - {nama}", OVERPASS, raw, "ODbL 1.0 (c) OpenStreetMap contributors", f"metode={metode}")
    for e in js["elements"]:
        e["kabupaten"] = nama; e["metode_wilayah"] = metode
    elems += js["elements"]
    print(nama, "->", len(js["elements"]), "elemen ("+metode+")")
    time.sleep(5)   # jeda sopan dinaikkan agar menghindari rate limit

df3 = pd.json_normalize(elems, sep="_")
df3.columns = [c.replace("tags_", "") for c in df3.columns]
print("Total baris hasil normalisasi:", len(df3), "| kolom:", df3.shape[1], "| >= 30 baris:", len(df3) >= 30)
print("\nLog status HTTP:"); display(pd.DataFrame([h for h in HTTP_LOG if h["label"].startswith("S3")]))

Mengambil data untuk Manggarai Barat...
Manggarai Barat: hasil area terlalu sedikit (0), mencoba fallback bounding box...
[S3-Manggarai Barat-bbox] HTTP 502 -> retry 1/5 dalam 2s
[S3-Manggarai Barat-bbox] HTTP 502 -> retry 2/5 dalam 4s
[S3-Manggarai Barat-bbox] HTTP 502 -> retry 3/5 dalam 8s
Manggarai Barat -> 716 elemen (bbox)
Mengambil data untuk Manggarai...
Manggarai: hasil area terlalu sedikit (0), mencoba fallback bounding box...
Manggarai -> 433 elemen (bbox)
Mengambil data untuk Manggarai Timur...
[S3-Manggarai Timur] HTTP 502 -> retry 1/5 dalam 2s
[S3-Manggarai Timur] HTTP 502 -> retry 2/5 dalam 4s
Manggarai Timur: hasil area terlalu sedikit (0), mencoba fallback bounding box...
Manggarai Timur -> 382 elemen (bbox)
Total baris hasil normalisasi: 1531 | kolom: 84 | >= 30 baris: True

Log status HTTP:


,label,percobaan,status
0,S3-Manggarai Barat,1,200
1,S3-Manggarai Barat-bbox,1,502
2,S3-Manggarai Barat-bbox,2,502
3,S3-Manggarai Barat-bbox,3,502
4,S3-Manggarai Barat-bbox,4,200
5,S3-Manggarai,1,200
6,S3-Manggarai-bbox,1,200
7,S3-Manggarai Timur,1,502
8,S3-Manggarai Timur,2,502
9,S3-Manggarai Timur,3,200


**pembersihan data OSM**

In [55]:
# Pembersihan -> data/processed. Kolom kontak/alamat (berpotensi PII) DIBUANG (minimisasi data).
pii_cols = [c for c in df3.columns if re.search(r"phone|email|contact|fax|whatsapp|addr", c, re.I)]
print("Kolom kontak/alamat dibuang:", pii_cols)
keep = [c for c in ["id", "lat", "lon", "name", "place", "tourism", "kabupaten", "metode_wilayah", "wikidata"] if c in df3.columns]
osm = df3.drop(columns=pii_cols, errors="ignore")[keep].copy()
osm = osm.rename(columns={"id": "osm_id", "name": "nama"})
osm = osm[osm["nama"].notna()].drop_duplicates(subset=["osm_id"]).reset_index(drop=True)
osm["kategori"] = np.where(osm["place"].notna(), "wilayah", "objek_wisata") if "place" in osm else "objek_wisata"
osm["key_kab"] = osm["kabupaten"].map(norm)
osm["key_desa"] = osm["key_kab"] + "|" + osm["nama"].map(norm)
PROC3 = f"{BASE}/data/processed/osm_manggarai_raya_bersih.csv"
osm.to_csv(PROC3, index=False, encoding="utf-8")
print("Baris bersih:", len(osm)); print(osm["kabupaten"].value_counts().to_string()); osm.head(10)

Kolom kontak/alamat dibuang: ['email', 'phone', 'addr:city', 'addr:postcode', 'addr:street', 'addr:housenumber', 'addr:country', 'contact:language', 'addr:district', 'addr:housename', 'addr:neighbourhood', 'addr:province', 'addr:subdistrict', 'addr:unit']
Baris bersih: 911
kabupaten
Manggarai Barat    669
Manggarai Timur    158
Manggarai           84


,osm_id,lat,lon,nama,place,tourism,kabupaten,metode_wilayah,wikidata,kategori,key_kab,key_desa
0,703787171,-8.482560,119.880461,Golo Hilltop,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|golo hilltop
1,727766254,-8.492503,119.877713,Gardena,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|gardena
2,834902092,-8.508675,119.877342,New Bajo,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|new bajo
3,931543319,-8.582300,119.800000,Pulau Purung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,manggarai barat,manggarai barat|pulau purung
4,931543800,-8.624800,119.806200,Pulau Gadung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,manggarai barat,manggarai barat|pulau gadung
5,1070187399,-8.467900,119.878000,Bukit Binongko,NaN,viewpoint,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|bukit binongko
6,1118552623,-8.611507,120.445643,Compang Ruteng,hamlet,NaN,Manggarai Barat,bbox,NaN,wilayah,manggarai barat,manggarai barat|compang ruteng
7,1118552624,-8.611880,120.446034,Compang Ruteng,NaN,yes,Manggarai Barat,bbox,NaN,objek_wisata,manggarai barat,manggarai barat|compang ruteng
8,1271854768,-8.606215,120.347281,Pong Leko,village,NaN,Manggarai Barat,bbox,Q24639006,wilayah,manggarai barat,manggarai barat|pong leko
9,1271854770,-8.457569,120.341388,Lumut,village,NaN,Manggarai Barat,bbox,Q24597086,wilayah,manggarai barat,manggarai barat|lumut


**cek robots.txt dan scraping tiga halaman Wikipedia**

In [56]:
PAGES4 = {"Manggarai Barat": "https://id.wikipedia.org/wiki/Kabupaten_Manggarai_Barat",
          "Manggarai": "https://id.wikipedia.org/wiki/Kabupaten_Manggarai",
          "Manggarai Timur": "https://id.wikipedia.org/wiki/Kabupaten_Manggarai_Timur"}
ROBOTS4 = "https://id.wikipedia.org/robots.txt"

rob = fetch(ROBOTS4, label="S4-robots")
RAW4R = f"{BASE}/data/raw/robots_id_wikipedia.txt"
open(RAW4R, "w", encoding="utf-8").write(rob.text)
log_source("robots.txt id.wikipedia.org", ROBOTS4, RAW4R, "Bukti izin crawling")
rp = robotparser.RobotFileParser(); rp.parse(rob.text.splitlines())

TABS, CANDS = {}, {}
for nama, url in PAGES4.items():
    boleh = rp.can_fetch(UA_NAME, url)
    print(f"{nama}: diizinkan scraping -> {boleh}")
    assert boleh, f"robots.txt melarang {url} - JANGAN scraping"
    time.sleep(2)
    pg = fetch(url, label=f"S4-{nama}")
    raw = f"{BASE}/data/raw/wikipedia_kab_{nama.lower().replace(' ', '_')}.html"
    open(raw, "w", encoding="utf-8").write(pg.text)
    log_source(f"Wikipedia ID - Kabupaten {nama}", url, raw, "CC BY-SA 4.0 (Wikipedia)", "Scraping statis; atribusi wajib")
    TABS[nama] = pd.read_html(io.StringIO(pg.text))
    CANDS[nama] = [i for i, t in enumerate(TABS[nama]) if any("kecamatan" in str(c).lower() for c in t.columns) and len(t) >= 5]
    print("   tabel kandidat:", [(i, TABS[nama][i].shape) for i in CANDS[nama]])

Manggarai Barat: diizinkan scraping -> True
   tabel kandidat: [(5, (16, 6))]
Manggarai: diizinkan scraping -> True
   tabel kandidat: [(5, (16, 6))]
Manggarai Timur: diizinkan scraping -> True
   tabel kandidat: [(6, (17, 6))]


**parsing tabel kecamatan**

In [57]:

# DIAGNOSTIK: PERIKSA SEMUA TABEL WIKIPEDIA

for nama in PAGES4:
    print("\n" + "=" * 65)
    print("KABUPATEN:", nama)
    print("JUMLAH TABEL:", len(TABS[nama]))

    for i, tabel in enumerate(TABS[nama]):
        kolom = tabel.columns

        if isinstance(kolom, pd.MultiIndex):
            nama_kolom = [
                " | ".join(str(x) for x in c if str(x) != "nan")
                for c in kolom
            ]
        else:
            nama_kolom = [str(c) for c in kolom]

        print(f"\nTabel indeks: {i}")
        print("Ukuran:", tabel.shape)
        print("Kolom:", nama_kolom)
        print("Contoh data:")
        display(tabel.head(3))


KABUPATEN: Manggarai Barat
JUMLAH TABEL: 11

Tabel indeks: 0
Ukuran: (33, 2)
Kolom: ['Kabupaten Manggarai Barat', 'Kabupaten Manggarai Barat.1']
Contoh data:


,Kabupaten Manggarai Barat,Kabupaten Manggarai Barat.1
0,Kabupaten,Kabupaten
1,"Pulau Padar, Taman Nasional Pulau Komodo","Pulau Padar, Taman Nasional Pulau Komodo"
2,Lambang,Lambang



Tabel indeks: 1
Ukuran: (4, 2)
Kolom: ['0', '1']
Contoh data:


,0,1
0,Utara,Laut Flores
1,Timur,Kabupaten Manggarai
2,Selatan,Selat Sumba dan Laut Sawu



Tabel indeks: 2
Ukuran: (9, 14)
Kolom: ['Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Bulan', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Jan', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Feb', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Mar', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Apr', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Mei', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Jun', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Jul', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Agt', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Sep', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Okt', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Nov', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Des', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Tahun']
Contoh data:


Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia                \
                                                   Bulan           Jan   
0                        Rata-rata tertinggi °C (°F)      29.2  (84.6)   
1                           Rata-rata harian °C (°F)      25.8  (78.4)   
2                         Rata-rata terendah °C (°F)      23.2  (73.8)   

                                                                         \
            Feb           Mar           Apr           Mei           Jun   
0  29.1  (84.4)  29.6  (85.3)  30.1  (86.2)  30.2  (86.4)  29.7  (85.5)   
1  25.6  (78.1)  25.7  (78.3)  25.7  (78.3)  25.5  (77.9)  24.9  (76.8)   
2  23.1  (73.6)  23.3  (73.9)  23.6  (74.5)  23.5  (74.3)    22.8  (73)   

                                                                         \
            Jul           Agt           Sep           Okt           Nov   
0  29.5  (85.1)  29.8  (85.6)  30.4  (86.7)  30.9  (87.6)      31  (88)   
1  24.2  (75.6)  24.4  (75.9)  25.3  (77.5)  26.3  (79.3)  26.7  (80.1)   
2    22.2  (72)      22  (72)  22.7  (72.9)  23.5  (74.3)      24  (75)   

                                 
            Des           Tahun  
0  30.3  (86.5)  29.98  (85.99)  
1    26.1  (79)  25.52  (77.93)  
2  23.5  (74.3)  23.12  (73.63)

... (54 outputs left out)


Tabel indeks: 9
Ukuran: (4, 2)
Kolom: ['lbsWilayah di Indonesia yang mayoritas beragama Kristen', 'lbsWilayah di Indonesia yang mayoritas beragama Kristen.1']
Contoh data:


,lbsWilayah di Indonesia yang mayoritas beragama Kristen,lbsWilayah di Indonesia yang mayoritas beragama Kristen.1
0,Provinsi,Nusa Tenggara Timur Papua Papua Barat Papua Ba...
1,Kabupaten,Alor Asmat Belu Bengkayang Biak Numfor Boven D...
2,Kota,Ambon Bitung Gunung Sitoli Jayapura Kupang Man...



Tabel indeks: 10
Ukuran: (3, 2)
Kolom: ['Basis data pengawasan otoritas', 'Basis data pengawasan otoritas.1']
Contoh data:


,Basis data pengawasan otoritas,Basis data pengawasan otoritas.1
0,Internasional,VIAF
1,Nasional,Amerika Serikat
2,Lain-lain,Yale LUX


In [58]:

# DIAGNOSTIK: PERIKSA SEMUA TABEL WIKIPEDIA

for nama in PAGES4:
    print("\n" + "=" * 65)
    print("KABUPATEN:", nama)
    print("JUMLAH TABEL:", len(TABS[nama]))

    for i, tabel in enumerate(TABS[nama]):
        kolom = tabel.columns

        if isinstance(kolom, pd.MultiIndex):
            nama_kolom = [
                " | ".join(str(x) for x in c if str(x) != "nan")
                for c in kolom
            ]
        else:
            nama_kolom = [str(c) for c in kolom]

        print(f"\nTabel indeks: {i}")
        print("Ukuran:", tabel.shape)
        print("Kolom:", nama_kolom)
        print("Contoh data:")
        display(tabel.head(3))


KABUPATEN: Manggarai Barat
JUMLAH TABEL: 11

Tabel indeks: 0
Ukuran: (33, 2)
Kolom: ['Kabupaten Manggarai Barat', 'Kabupaten Manggarai Barat.1']
Contoh data:


,Kabupaten Manggarai Barat,Kabupaten Manggarai Barat.1
0,Kabupaten,Kabupaten
1,"Pulau Padar, Taman Nasional Pulau Komodo","Pulau Padar, Taman Nasional Pulau Komodo"
2,Lambang,Lambang



Tabel indeks: 1
Ukuran: (4, 2)
Kolom: ['0', '1']
Contoh data:


,0,1
0,Utara,Laut Flores
1,Timur,Kabupaten Manggarai
2,Selatan,Selat Sumba dan Laut Sawu



Tabel indeks: 2
Ukuran: (9, 14)
Kolom: ['Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Bulan', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Jan', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Feb', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Mar', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Apr', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Mei', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Jun', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Jul', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Agt', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Sep', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Okt', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Nov', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Des', 'Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia | Tahun']
Contoh data:


Data iklim Labuan Bajo, Nusa Tenggara Timur, Indonesia                \
                                                   Bulan           Jan   
0                        Rata-rata tertinggi °C (°F)      29.2  (84.6)   
1                           Rata-rata harian °C (°F)      25.8  (78.4)   
2                         Rata-rata terendah °C (°F)      23.2  (73.8)   

                                                                         \
            Feb           Mar           Apr           Mei           Jun   
0  29.1  (84.4)  29.6  (85.3)  30.1  (86.2)  30.2  (86.4)  29.7  (85.5)   
1  25.6  (78.1)  25.7  (78.3)  25.7  (78.3)  25.5  (77.9)  24.9  (76.8)   
2  23.1  (73.6)  23.3  (73.9)  23.6  (74.5)  23.5  (74.3)    22.8  (73)   

                                                                         \
            Jul           Agt           Sep           Okt           Nov   
0  29.5  (85.1)  29.8  (85.6)  30.4  (86.7)  30.9  (87.6)      31  (88)   
1  24.2  (75.6)  24.4  (75.9)  25.3  (77.5)  26.3  (79.3)  26.7  (80.1)   
2    22.2  (72)      22  (72)  22.7  (72.9)  23.5  (74.3)      24  (75)   

                                 
            Des           Tahun  
0  30.3  (86.5)  29.98  (85.99)  
1    26.1  (79)  25.52  (77.93)  
2  23.5  (74.3)  23.12  (73.63)

... (54 outputs left out)


Tabel indeks: 9
Ukuran: (4, 2)
Kolom: ['lbsWilayah di Indonesia yang mayoritas beragama Kristen', 'lbsWilayah di Indonesia yang mayoritas beragama Kristen.1']
Contoh data:


,lbsWilayah di Indonesia yang mayoritas beragama Kristen,lbsWilayah di Indonesia yang mayoritas beragama Kristen.1
0,Provinsi,Nusa Tenggara Timur Papua Papua Barat Papua Ba...
1,Kabupaten,Alor Asmat Belu Bengkayang Biak Numfor Boven D...
2,Kota,Ambon Bitung Gunung Sitoli Jayapura Kupang Man...



Tabel indeks: 10
Ukuran: (3, 2)
Kolom: ['Basis data pengawasan otoritas', 'Basis data pengawasan otoritas.1']
Contoh data:


,Basis data pengawasan otoritas,Basis data pengawasan otoritas.1
0,Internasional,VIAF
1,Nasional,Amerika Serikat
2,Lain-lain,Yale LUX


**integrasi (merge + validate + cek duplikasi)**

In [59]:
A = df1_raya.copy()
A["key_kecamatan"] = A["key_kab"] + "|" + A["kecamatan"].map(norm)
A["key_desa"] = A["key_kab"] + "|" + A["desa_kelurahan"].map(norm)
A = A.drop_duplicates(subset=["key_desa", "key_kecamatan"])
A.to_csv(f"{BASE}/data/processed/kemenpar_manggarai_raya_bersih.csv", index=False, encoding="utf-8")

# Merge 1: A x C (banyak desa -> satu kecamatan)
AC = A.merge(t4, on="key_kecamatan", how="left", validate="m:1", suffixes=("", "_wiki"), indicator="_m_kec")
print("Merge A x C:\n", AC["_m_kec"].value_counts().to_string())
AC = AC.drop(columns="_m_kec")

# Merge 2: B (OSM) x AC
AC_u = AC.drop_duplicates(subset=["key_desa"])
final = osm.merge(AC_u, on="key_desa", how="left", validate="m:1", suffixes=("", "_kemenpar"), indicator="_m_desa")
final["status_desa_wisata_kemenpar"] = np.where(final["_m_desa"] == "both", "tersertifikasi", "tidak_tercatat")
print("\nMerge OSM x (A+C):\n", final["_m_desa"].value_counts().to_string())
final = final.drop(columns="_m_desa")

print("\nDuplikat baris penuh:", int(final.duplicated().sum()), "| duplikat osm_id:", int(final["osm_id"].duplicated().sum()))
final = final.drop_duplicates().reset_index(drop=True)
OUT = f"{BASE}/data/output/desa_wisata_manggarai_raya_terintegrasi.csv"
final.to_csv(OUT, index=False, encoding="utf-8")
print("Hasil akhir:", final.shape); print(final.groupby(["kabupaten", "status_desa_wisata_kemenpar"]).size().to_string()); final.head()

Merge A x C:
 _m_kec
left_only     0
right_only    0
both          0

Merge OSM x (A+C):
 _m_desa
left_only     911
right_only      0
both            0

Duplikat baris penuh: 0 | duplikat osm_id: 0
Hasil akhir: (911, 36)
kabupaten        status_desa_wisata_kemenpar
Manggarai        tidak_tercatat                  84
Manggarai Barat  tidak_tercatat                 669
Manggarai Timur  tidak_tercatat                 158


,osm_id,lat,lon,nama,place,tourism,kabupaten,metode_wilayah,wikidata,kategori,...,key_kecamatan,kode_kemendagri,kecamatan_wiki,jumlah_kelurahan,jumlah_desa,status,daftar_desa/kelurahan,kabupaten_wiki,key_kab_wiki,status_desa_wisata_kemenpar
0,703787171,-8.482560,119.880461,Golo Hilltop,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,...,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
1,727766254,-8.492503,119.877713,Gardena,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,...,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
2,834902092,-8.508675,119.877342,New Bajo,NaN,hotel,Manggarai Barat,bbox,NaN,objek_wisata,...,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
3,931543319,-8.582300,119.800000,Pulau Purung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,...,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat
4,931543800,-8.624800,119.806200,Pulau Gadung,island,NaN,Manggarai Barat,bbox,NaN,wilayah,...,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,tidak_tercatat


**metadata dan provenance**

In [60]:
# Metadata + dokumentasi provenance + checklist etika + README
meta = {
    "judul": "Dataset Terintegrasi Desa Wisata Manggarai Raya (Manggarai, Manggarai Timur, Manggarai Barat)",
    "dibuat": TODAY, "jumlah_baris": int(len(final)), "jumlah_kolom": int(final.shape[1]),
    "kolom": list(final.columns), "kunci_merge": ["key_kecamatan", "key_desa"],
    "sumber": [{k: p[k] for k in ("nama", "url", "lisensi", "md5")} for p in PROV],
    "md5_output": md5_file(OUT),
}
json.dump(meta, open(f"{BASE}/data/output/metadata.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
log_source("Hasil integrasi (output akhir)", "(dihasilkan notebook ini)", OUT, "Mengikuti lisensi sumber (ODbL & CC BY-SA -> share-alike)")

prov = pd.DataFrame(PROV)
with open(f"{BASE}/docs/provenance.md", "w", encoding="utf-8") as f:
    f.write("# Provenance Data\n\n| Sumber | URL | Tanggal akses | Lisensi | File | MD5 |\n|---|---|---|---|---|---|\n")
    for p in PROV:
        f.write(f"| {p['nama']} | {p['url']} | {p['tanggal_akses']} | {p['lisensi']} | `{p['file']}` | `{p['md5']}` |\n")
prov[["nama", "tanggal_akses", "md5"]]

,nama,tanggal_akses,md5
0,Kemenparekraf - Desa Wisata Berkelanjutan Ters...,2026-11-26,839784cd31adbe1ebb94831cd0cf0def
1,OpenStreetMap via Overpass - Manggarai Barat,2026-11-26,d0fd9362ee9100078f4b373f90625b58
2,OpenStreetMap via Overpass - Manggarai,2026-11-26,abf8e24769df4c18699670f7ce4b2d98
3,OpenStreetMap via Overpass - Manggarai Timur,2026-11-26,28adc5d5754fdbc0a8d65711e2004d74
4,robots.txt id.wikipedia.org,2026-11-26,8dd94bf3d01cbb25e0af4e64c0485433
5,Wikipedia ID - Kabupaten Manggarai Barat,2026-11-26,d7b43764a49e70a103ecff31983ae7b3
6,Wikipedia ID - Kabupaten Manggarai,2026-11-26,e033c9152b2098e6ca30184b6b497f96
7,Wikipedia ID - Kabupaten Manggarai Timur,2026-11-26,7a790b0f071d15a696688e96036810f9
8,Hasil integrasi (output akhir),2026-11-26,f0053bfcdb997550bd62ff726f53ba48


**Data Ethics Checklist 9 poin**

In [61]:
pii_all = pd.concat([pii_s1, scan_pii(final, "Output-akhir")], ignore_index=True)
temuan = "tidak ada temuan" if pii_all.empty else f"{len(pii_all)} temuan - periksa tabel di atas"
checklist = [
 ("1. Sumber & kepemilikan data jelas", "Kemenpar (pemerintah), OpenStreetMap contributors, Wikipedia (3 artikel kabupaten) - URL ada di docs/provenance.md"),
 ("2. Lisensi diperiksa", "Kemenpar: lisensi tidak dinyatakan pada portal (tangkapan halaman, tgl akses " + TODAY + "); OSM: ODbL 1.0; Wikipedia: CC BY-SA 4.0"),
 ("3. Ketentuan portal/ToS dipatuhi", "Overpass: 3 request kecil + jeda + retry/backoff; Wikipedia: robots.txt dicek (file raw tersimpan), User-Agent jelas"),
 ("4. Pemeriksaan data pribadi (PII)", "Pemindaian regex email/telepon/NIK: " + temuan + "; kolom kontak/alamat OSM dibuang"),
 ("5. Quasi-identifier dinilai", "Kolom wilayah/tahun saja: " + ", ".join(QUASI) + " - level entitas wilayah, bukan individu"),
 ("6. Minimisasi data", "Hanya kolom yang dibutuhkan disimpan pada data/processed"),
 ("7. Penyimpanan & integritas", "MD5 setiap file raw dicatat di docs/provenance.md; raw tidak diubah"),
 ("8. Atribusi & transparansi", "Atribusi sumber di README; hasil gabungan mengikuti ODbL/CC BY-SA (share-alike)"),
 ("9. Dampak & penggunaan wajar", "Untuk tujuan akademik; tidak ada profiling individu; scraping 3 halaman dengan jeda 2 detik"),
]
with open(f"{BASE}/docs/data_ethics_checklist.md", "w", encoding="utf-8") as f:
    f.write("# Data Ethics Checklist\n\n| Butir | Bukti |\n|---|---|\n")
    for a, b in checklist: f.write(f"| {a} | {b} |\n")
pd.DataFrame(checklist, columns=["Butir", "Bukti"])

,Butir,Bukti
0,1. Sumber & kepemilikan data jelas,"Kemenpar (pemerintah), OpenStreetMap contribut..."
1,2. Lisensi diperiksa,Kemenpar: lisensi tidak dinyatakan pada portal...
2,3. Ketentuan portal/ToS dipatuhi,Overpass: 3 request kecil + jeda + retry/backo...
3,4. Pemeriksaan data pribadi (PII),Pemindaian regex email/telepon/NIK: tidak ada ...
4,5. Quasi-identifier dinilai,Kolom wilayah/tahun saja: tahun_awal_sertifika...
5,6. Minimisasi data,Hanya kolom yang dibutuhkan disimpan pada data...
6,7. Penyimpanan & integritas,MD5 setiap file raw dicatat di docs/provenance...
7,8. Atribusi & transparansi,Atribusi sumber di README; hasil gabungan meng...
8,9. Dampak & penggunaan wajar,Untuk tujuan akademik; tidak ada profiling ind...


In [62]:
print("Jumlah data per kabupaten:")
print(t4["kabupaten"].value_counts())

print("\nJumlah kecamatan unik:")
print(t4[["kabupaten", "kecamatan"]].drop_duplicates().shape[0])

print("\nContoh data:")
display(t4.head(10))

print("\nJumlah data akhir:")
print(t4.shape)

Jumlah data per kabupaten:
kabupaten
Manggarai Barat    12
Manggarai          12
Manggarai Timur    10
Name: count, dtype: int64

Jumlah kecamatan unik:
34

Contoh data:


,kode_kemendagri,kecamatan,jumlah_kelurahan,jumlah_desa,status,daftar_desa/kelurahan,kabupaten,key_kab,key_kecamatan
0,53.15.01,Macang Pacar,NaN,13,Desa,Bari Lewat Mbakung Nanga Kantor Nanga Kantor B...,Manggarai Barat,manggarai barat,manggarai barat|macang pacar
1,53.15.02,Kuwus,2.0,10,Desa,Bangka Lewat Benteng Suru Coal Compang Suka Go...,Manggarai Barat,manggarai barat,manggarai barat|kuwus
2,53.15.03,Lembor,1.0,14,Desa,Daleng Golo Ndeweng Liang Sola Ngancar Poco De...,Manggarai Barat,manggarai barat,manggarai barat|lembor
3,53.15.04,Sano Nggoang,NaN,15,Desa,Golo Kempo Golo Kondeng Golo Leleng Golo Manti...,Manggarai Barat,manggarai barat,manggarai barat|sano nggoang
4,53.15.05,Komodo,2.0,17,Desa,Batu Cermin Compang Longgo Golo Bilas Golo Mor...,Manggarai Barat,manggarai barat,manggarai barat|komodo
5,53.15.06,Boleng,NaN,11,Desa,Batu Tiga Beo Sepang Golo Ketak Golo Lujang Go...,Manggarai Barat,manggarai barat,manggarai barat|boleng
6,53.15.07,Welak,NaN,16,Desa,Dunta Galang Golo Ndari Golo Ronggot Gurung La...,Manggarai Barat,manggarai barat,manggarai barat|welak
7,53.15.08,Ndoso,NaN,15,Desa,Golo Bore Golo Keli Golo Poleng Golo Ru'a Lumu...,Manggarai Barat,manggarai barat,manggarai barat|ndoso
8,53.15.09,Lembor Selatan,NaN,15,Desa,Benteng Dewa Benteng Tado Kakor Lalong Lendong...,Manggarai Barat,manggarai barat,manggarai barat|lembor selatan
9,53.15.10,Mbeliling,NaN,15,Desa,Compang Liang Ndara Cunca Lolos Cunca Wulang G...,Manggarai Barat,manggarai barat,manggarai barat|mbeliling



Jumlah data akhir:
(34, 9)


In [63]:
import os

# Membuat folder hasil
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/output", exist_ok=True)

print("Folder berhasil dibuat!")

Folder berhasil dibuat!


In [64]:
import pandas as pd

# Menampilkan semua DataFrame yang sudah dibuat
for nama, data in globals().copy().items():
    if isinstance(data, pd.DataFrame):
        print(f"{nama} : {data.shape}")

_ : (9, 2)
__ : (9, 3)
___ : (5, 36)
df1 : (10, 13)
_2 : (5, 12)
df1_raya : (0, 14)
est : (2, 3)
_6 : (2, 3)
_7 : (2, 3)
pii_s1 : (0, 4)
_9 : (2, 3)
df3 : (1531, 84)
osm : (911, 12)
_12 : (10, 12)
t : (16, 7)
t4 : (34, 9)
_14 : (10, 9)
_15 : (10, 9)
A : (0, 16)
AC : (0, 24)
AC_u : (0, 24)
final : (911, 36)
_16 : (5, 36)
prov : (9, 8)
_17 : (9, 3)
pii_all : (0, 4)
_18 : (9, 2)
temp : (17, 6)
_21 : (5, 36)
_22 : (10, 3)
_23 : (9, 2)
tabel : (3, 2)
_32 : (2, 3)
_35 : (10, 12)
_39 : (5, 36)
_40 : (9, 3)
_41 : (9, 2)
_52 : (2, 3)
_55 : (10, 12)
_59 : (5, 36)
_60 : (9, 3)
_61 : (9, 2)


In [65]:
df_osm : (911, 10)
df_gabungan : (911, 12)

In [67]:
import pandas as pd

daftar_df = {
    nama: variabel.shape
    for nama, variabel in globals().copy().items()
    if isinstance(variabel, pd.DataFrame)
}

print("DAFTAR DATAFRAME:")
for nama, ukuran in daftar_df.items():
    print(f"{nama}: {ukuran}")

DAFTAR DATAFRAME:
_: (9, 2)
__: (9, 3)
___: (5, 36)
df1: (10, 13)
_2: (5, 12)
df1_raya: (0, 14)
est: (2, 3)
_6: (2, 3)
_7: (2, 3)
pii_s1: (0, 4)
_9: (2, 3)
df3: (1531, 84)
osm: (911, 12)
_12: (10, 12)
t: (16, 7)
t4: (34, 9)
_14: (10, 9)
_15: (10, 9)
A: (0, 16)
AC: (0, 24)
AC_u: (0, 24)
final: (911, 36)
_16: (5, 36)
prov: (9, 8)
_17: (9, 3)
pii_all: (0, 4)
_18: (9, 2)
temp: (17, 6)
_21: (5, 36)
_22: (10, 3)
_23: (9, 2)
tabel: (3, 2)
_32: (2, 3)
_35: (10, 12)
_39: (5, 36)
_40: (9, 3)
_41: (9, 2)
_52: (2, 3)
_55: (10, 12)
_59: (5, 36)
_60: (9, 3)
_61: (9, 2)


In [68]:
import os

os.makedirs("data/processed", exist_ok=True)

final.to_csv(
    "data/processed/hasil_pengolahan.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Berhasil menyimpan hasil pengolahan data!")
print("Jumlah baris:", len(final))
print("Jumlah kolom:", len(final.columns))

Berhasil menyimpan hasil pengolahan data!
Jumlah baris: 911
Jumlah kolom: 36


In [71]:
from google.colab import files

files.download("UTS_DAM_2501010326.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [72]:
import os
from getpass import getpass
from pathlib import Path

GITHUB_USER = input("Username GitHub: ").strip()
GITHUB_TOKEN = getpass("GitHub token (tidak akan ditampilkan): ")

REPO_URL = f"https://github.com/{GITHUB_USER}/UTS_DAM_2501010326.git"

Username GitHub: UTS_DAM_2501010326
GitHub token (tidak akan ditampilkan): ··········


In [73]:
!git clone {REPO_URL} /content/UTS_DAM_2501010326

Cloning into '/content/UTS_DAM_2501010326'...
fatal: could not read Username for 'https://github.com': No such device or address


In [75]:
import shutil
from pathlib import Path

sumber = Path("/content/data")
tujuan = Path("/content/UTS_DAM_2501010326/data")

if sumber.exists():
    shutil.copytree(sumber, tujuan, dirs_exist_ok=True)
    print("Folder data berhasil disalin ke repository.")
else:
    print("Folder sumber tidak ditemukan:", sumber)

Folder data berhasil disalin ke repository.


In [76]:
!ls -la /content
!ls -la /content/data

total 7240
drwxr-xr-x 1 root root    4096 Nov 26 20:00 .
drwxr-xr-x 1 root root    4096 Nov 26 16:37 ..
drwxr-xr-x 4 root root    4096 Nov 18 15:58 .config
drwxr-xr-x 5 root root    4096 Nov 26 19:14 data
drwxr-xr-x 1 root root    4096 Nov 18 15:58 sample_data
drwxr-xr-x 5 root root    4096 Nov 26 17:57 UTS-DAM
drwxr-xr-x 3 root root    4096 Nov 26 20:00 UTS_DAM_2501010326
-rw-r--r-- 1 root root 7382781 Nov 26 19:41 UTS_DAM_2501010326.zip
total 20
drwxr-xr-x 5 root root 4096 Nov 26 19:14 .
drwxr-xr-x 1 root root 4096 Nov 26 20:00 ..
drwxr-xr-x 2 root root 4096 Nov 26 19:14 output
drwxr-xr-x 2 root root 4096 Nov 26 19:35 processed
drwxr-xr-x 2 root root 4096 Nov 26 19:14 raw


In [77]:
%cd /content/UTS_DAM_2501010326

!git config user.name "Nama GitHub kamu"
!git config user.email "Email GitHub kamu"

!git status

/content/UTS_DAM_2501010326
fatal: not in a git directory
fatal: not in a git directory
fatal: not a git repository (or any of the parent directories): .git


In [78]:
!git add data
!git commit -m "Menambahkan dan memperbarui data"

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [81]:
%cd /content/UTS_DAM_2501010326

!git add -A
!git status

/content/UTS_DAM_2501010326
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [82]:
!git commit -m "Upload seluruh file project"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


In [83]:
!git branch --show-current

fatal: not a git repository (or any of the parent directories): .git


In [84]:
GITHUB_TOKEN = None